In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ast import literal_eval

In [7]:
def safe_literal_eval(x):
    """Safely evaluate string representations of lists with error handling"""
    try:
        if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
            return literal_eval(x)
        return []
    except (ValueError, SyntaxError):
        return []

In [8]:
def safe_stats(data):
    """Calculate statistics safely handling empty/NaN data"""
    if len(data) == 0 or all(np.isnan(x) for x in data):
        return {
            'min': np.nan,
            'max': np.nan,
            'mean': np.nan,
            'std': np.nan,
            'peak': np.nan
        }
    clean_data = [x for x in data if not np.isnan(x)]
    if not clean_data:
        return {
            'min': np.nan,
            'max': np.nan,
            'mean': np.nan,
            'std': np.nan,
            'peak': np.nan
        }
    return {
        'min': np.nanmin(clean_data),
        'max': np.nanmax(clean_data),
        'mean': np.nanmean(clean_data),
        'std': np.nanstd(clean_data),
        'peak': clean_data[np.nanargmax(clean_data)] if clean_data else np.nan
    }


In [9]:
def section_analysis(df):
    """Perform robust section-wise analysis on current signals"""
    results = []
    
    for idx, row in df.iterrows():
        try:
            # Safely convert string representations of lists
            current_a = safe_literal_eval(row['A Current'])
            current_b = safe_literal_eval(row['B Current'])
            voltage_a = safe_literal_eval(row['A Voltage'])
            voltage_b = safe_literal_eval(row['B Voltage'])
            
            # Skip if any signal is empty or all NaN
            if (not current_a or not current_b or not voltage_a or not voltage_b or
                all(np.isnan(x) for x in current_a) or all(np.isnan(x) for x in current_b)):
                continue
                
            # Create analysis record
            record_analysis = {
                'Time': row['Time'],
                'Site Name': row['Site Name'],
                'Point Machine Name': row['Point Machine Name'],
                'Direction': row['Direction'],
                'Total Samples A': len(current_a),
                'Total Samples B': len(current_b)
            }
            
            # Define section boundaries (4 equal sections)
            sections = 4
            a_section_size = max(1, len(current_a) // sections)
            b_section_size = max(1, len(current_b) // sections)
            
            # Analyze each section for Current A
            for i in range(sections):
                start = i * a_section_size
                end = min((i + 1) * a_section_size, len(current_a)) if i < sections - 1 else len(current_a)
                
                section_data = current_a[start:end]
                section_voltage = voltage_a[start:end]
                
                # Calculate stats safely
                current_stats = safe_stats(section_data)
                voltage_stats = safe_stats(section_voltage)
                
                record_analysis.update({
                    f'A_Section_{i+1}_Start': start,
                    f'A_Section_{i+1}_End': end - 1,
                    f'A_Section_{i+1}_Min': current_stats['min'],
                    f'A_Section_{i+1}_Max': current_stats['max'],
                    f'A_Section_{i+1}_Mean': current_stats['mean'],
                    f'A_Section_{i+1}_Std': current_stats['std'],
                    f'A_Section_{i+1}_Voltage_Mean': voltage_stats['mean'],
                    f'A_Section_{i+1}_Peak_Current': current_stats['peak'],
                    f'A_Section_{i+1}_Peak_Voltage': voltage_stats['peak']
                })
            
            # Analyze each section for Current B
            for i in range(sections):
                start = i * b_section_size
                end = min((i + 1) * b_section_size, len(current_b)) if i < sections - 1 else len(current_b)
                
                section_data = current_b[start:end]
                section_voltage = voltage_b[start:end]
                
                # Calculate stats safely
                current_stats = safe_stats(section_data)
                voltage_stats = safe_stats(section_voltage)
                
                record_analysis.update({
                    f'B_Section_{i+1}_Start': start,
                    f'B_Section_{i+1}_End': end - 1,
                    f'B_Section_{i+1}_Min': current_stats['min'],
                    f'B_Section_{i+1}_Max': current_stats['max'],
                    f'B_Section_{i+1}_Mean': current_stats['mean'],
                    f'B_Section_{i+1}_Std': current_stats['std'],
                    f'B_Section_{i+1}_Voltage_Mean': voltage_stats['mean'],
                    f'B_Section_{i+1}_Peak_Current': current_stats['peak'],
                    f'B_Section_{i+1}_Peak_Voltage': voltage_stats['peak']
                })
            
            results.append(record_analysis)
            
        except Exception as e:
            print(f"Error processing row {idx}: {str(e)}")
            continue
    
    return pd.DataFrame(results)


In [10]:
def main():
    # Load your dataset
    df = pd.read_csv('/Users/anamikasaroha/Energy7_Week1/week_1/cleaned_WR.csv')
    
    # Perform section analysis
    analysis_results = section_analysis(df)
    
    # Save results to CSV
    analysis_results.to_csv('section_analysis_results.csv', index=False)
    
    print(f"Analysis complete. Processed {len(analysis_results)} records.")
    print(f"{len(df) - len(analysis_results)} records were skipped due to empty/malformed data.")

if __name__ == "__main__":
    main()

Analysis complete. Processed 32775 records.
0 records were skipped due to empty/malformed data.
